# Direct Generation Baseline for Omnilex Legal Retrieval

This notebook implements a **direct generation approach** where we prompt a local LLM to generate Swiss legal citations based on the query.

## Approach
1. Load a local LLM (GGUF format via llama-cpp-python)
2. For each query, prompt the LLM to directly generate relevant citations
3. Parse and normalize the generated citations
4. Create submission file

## Requirements
- llama-cpp-python (optional - runs in mock mode if not installed)
- A GGUF model file (optional - uses mock responses for demonstration if not available)
- pandas, tqdm

**Note**: This notebook runs in "mock mode" if llama-cpp-python is not installed or no model file is available, generating placeholder citations for demonstration purposes.

## 1. Setup & Configuration

In [1]:
import os
import sys
from pathlib import Path

from lexcite.config import ROOT_DIR, RAW_DATA_DIR, MODEL_DIR, OUTPUT_DIR

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "val"  # Change to "test" for final submission

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/omnilex-data")
    MODEL_PATH = Path("/kaggle/input/llama-model")
    OUTPUT_PATH = Path("/kaggle/working")
    sys.path.insert(0, "/kaggle/input/omnilex-utils")
else:
    # Local development paths
    REPO_ROOT = ROOT_DIR
    DATA_PATH = RAW_DATA_DIR
    MODEL_PATH = MODEL_DIR
    OUTPUT_PATH = OUTPUT_DIR
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
# OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Data path: {DATA_PATH}")
print(f"Model path: {MODEL_PATH}")
print(f"Output path: {OUTPUT_PATH}")

Environment: Local
Dataset mode: val
Query file: /workspaces/lexcite/data/raw/val.csv
Validation mode: True
Data path: /workspaces/lexcite/data/raw
Model path: /workspaces/lexcite/models
Output path: /workspaces/lexcite/output


In [2]:
# Configuration
CONFIG = {
    # Model settings
    "model_file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",  # Adjust to your model
    "n_ctx": 4096,         # Context window
    "n_threads": 4,        # CPU threads
    "n_gpu_layers": -1,    # GPU layers (-1 = offload all layers to GPU)
    
    # Generation settings
    "max_tokens": 512,
    "temperature": 0.0,    # Low temperature for consistency
    
    # Paths
    "test_file": "test.csv",
    "train_file": "train.csv",  # For local evaluation
}

## 2. Load Local LLM

In [3]:
from lexcite.llm_loader import load_model

from lexcite.config import GITHUB_MODEL_API, GITHUB_TOKEN

llm = load_model(
    provider="chatopenai",
    model="gpt-4.1",
    base_url=GITHUB_MODEL_API,
    api_key=GITHUB_TOKEN,
    temperature=0,   # Prefer high-probability tokens. Suitable for RAG tasks
    max_tokens=512,  # Limit response length. Suitable for our tasks
)

## 3. Define Generation Prompt

In [4]:
SYSTEM_PROMPT = """You are a Swiss legal citation expert. Output ONLY a Python list of citations.

CITATION FORMATS:
- Federal laws: "Art. X ABBREV" where ABBREV is ZGB, OR, StGB, BV, etc.
- Court decisions: "BGE X Y Z" or "BGE X Y Z E. N" with consideration number

OUTPUT FORMAT: Python list like ["citation1", "citation2", ...]

EXAMPLES:

Query: What are the requirements for a valid contract under Swiss law?
["Art. 1 OR", "Art. 11 OR", "Art. 12 OR", "BGE 119 II 449 E. 2", "BGE 127 III 248 E. 3.1"]

Query: When can a marriage be annulled in Switzerland?
["Art. 104 ZGB", "Art. 105 ZGB", "Art. 106 ZGB", "BGE 121 III 38 E. 2b"]

Query: What constitutes negligent homicide under Swiss criminal law?
["Art. 117 StGB", "Art. 12 StGB", "BGE 116 IV 306 E. 1a"]

Query: What are the grounds for divorce in Swiss law?
["Art. 111 ZGB", "Art. 112 ZGB", "Art. 114 ZGB", "Art. 115 ZGB", "BGE 130 III 585 E. 2.1"]

Query: How is inheritance distributed under Swiss law?
["Art. 457 ZGB", "Art. 462 ZGB", "Art. 471 ZGB", "BGE 132 III 305 E. 3.2"]

Now answer:"""

# Common Swiss law abbreviations for regex matching
LAW_ABBREVS = (
    "ZGB|OR|StGB|BV|SchKG|ZPO|StPO|BGG|VwVG|IPRG|KG|DSG|MSchG|URG|PatG|"
    "DesG|UWG|PrSG|FINMAG|BankG|VAG|KAG|GwG|BEHG|FinfraG|FIDLEG|FINIG|"
    "ATSG|AHV|IV|EO|ALV|KVG|UVG|BVG|ArG|GlG|USG|RPG|WaG|JSG|TSchG|"
    "LwG|PBG|EBG|SVG|LFG|SebG|SpG|BoeB|EMRK|SR|AS|BBl|ParlG|RVOG|RVOV|"
    "MG|BPG|BPV|VBGÖ|VDSG|MWSTG|DBG|StHG|VStG|StG|ZG|CO|CP|CC|CPC|CPP|"
    "LEtr|LAsi|LN|LDIP|LCart|LDA|LPM|LBI|LDes|LCD|LFINMA|LB|LSA|LPCC|"
    "LBA|LBVM|LIMF|LSFin|LEFin|LAVS|LAI|LAPG|LACI|LAMal|LAA|LPP|LTr|"
    "LEg|LPD|LPE|LAT|LFo|LChP|LPN|LAgr|LTV|LCdF|LNA|LPTh|LTAF|LTF"
)


def extract_citations(raw_output: str) -> list[str]:
    """Extract citations from raw LLM output using regex patterns."""
    import re

    citations = []

    # Pattern for BGE citations: BGE 141 II 345 E. 3.2
    # Matches: BGE + volume + part (roman) + page + optional consideration
    bge_pattern = r'BGE\s+(\d+)\s+([IVX]+[a-z]?)\s+(\d+)(?:\s+E\.\s*([\d.a-z/]+))?'
    for match in re.finditer(bge_pattern, raw_output):
        vol, part, page, consid = match.groups()
        if consid:
            citations.append(f"BGE {vol} {part} {page} E. {consid}")
        else:
            citations.append(f"BGE {vol} {part} {page}")

    # Pattern for Art. citations with Abs./lit./Ziff.
    # Matches: Art. 221 Abs. 1 lit. b StPO, Art. 364 Abs. 1 OR, Art. 1 ZGB
    art_pattern = rf'Art\.?\s*(\d+[a-z]?)(?:\s+(Abs\.?\s*\d+))?(?:\s+(lit\.?\s*[a-z]))?(?:\s+(Ziff\.?\s*\d+))?\s+({LAW_ABBREVS})\b'
    for match in re.finditer(art_pattern, raw_output, re.IGNORECASE):
        art_num, abs_part, lit_part, ziff_part, abbrev = match.groups()
        parts = [f"Art. {art_num}"]
        if abs_part:
            # Normalize "Abs1" or "Abs 1" to "Abs. 1"
            abs_normalized = re.sub(r'Abs\.?\s*', 'Abs. ', abs_part)
            parts.append(abs_normalized.strip())
        if lit_part:
            lit_normalized = re.sub(r'lit\.?\s*', 'lit. ', lit_part)
            parts.append(lit_normalized.strip())
        if ziff_part:
            ziff_normalized = re.sub(r'Ziff\.?\s*', 'Ziff. ', ziff_part)
            parts.append(ziff_normalized.strip())
        parts.append(abbrev.upper())
        citations.append(" ".join(parts))

    # Remove duplicates while preserving order
    seen = set()
    unique = []
    for c in citations:
        if c not in seen:
            seen.add(c)
            unique.append(c)

    return unique


def generate_citations(query: str) -> list[str]:
    """Generate citations using direct LLM prompting."""
    if llm is None:
        raise RuntimeError("LLM not loaded - please ensure model is available")
    
    from langchain_core.prompts import ChatPromptTemplate

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "Query: {query}")
    ])

    chain = prompt | llm
    response = chain.invoke({"query": query})

    raw_output: str = response.content.strip()  # type: ignore[index]
    print(f"Raw output: {raw_output}\n")

    # Parse Python list format first
    import re
    import ast
    
    citations = []
    
    # Try to parse as Python list
    try:
        list_match = re.search(r'\[.*?\]', raw_output, re.DOTALL)
        if list_match:
            parsed = ast.literal_eval(list_match.group())
            if isinstance(parsed, list):
                # Extract citations from parsed list items
                for item in parsed:
                    item_str = str(item).strip()
                    # Extract citations from each item (may have descriptions in parens)
                    extracted = extract_citations(item_str)
                    citations.extend(extracted)
    except (ValueError, SyntaxError):
        pass
    
    # If no citations found from list parsing, try extracting from full output
    if not citations:
        citations = extract_citations(raw_output)

    return citations

In [5]:
# Test generation with a sample query
test_query = "What are the requirements for a valid contract under Swiss law?"
print(f"Query: {test_query}")
raw_citations = generate_citations(test_query)
print("\nGenerated citations:")
for c in raw_citations:
    print(f"  - {c}")

Query: What are the requirements for a valid contract under Swiss law?
Raw output: ["Art. 1 OR", "Art. 11 OR", "Art. 12 OR", "BGE 119 II 449 E. 2", "BGE 127 III 248 E. 3.1"]


Generated citations:
  - Art. 1 OR
  - Art. 11 OR
  - Art. 12 OR
  - BGE 119 II 449 E. 2
  - BGE 127 III 248 E. 3.1


## 4. Load Test Data

In [6]:
import pandas as pd

# Load queries from the configured query file
if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

test_df = pd.read_csv(QUERY_FILE)

print(f"Loaded {len(test_df)} queries from {QUERY_FILE}")
print(f"Columns: {list(test_df.columns)}")

if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    print(f"Gold citations available for evaluation")

test_df.head()

Loaded 10 queries from /workspaces/lexcite/data/raw/val.csv
Columns: ['query_id', 'query', 'gold_citations']
Gold citations available for evaluation


,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


## 5. Generate Predictions

In [7]:
from tqdm.notebook import tqdm

# Generate predictions
predictions = []

assert test_df is not None, "test_df must be loaded before generating predictions"

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating predictions"):
    query_id = row["query_id"]
    query_text = row["query"]

    # Generate citations using regex extraction (no external normalizer needed)
    raw_citations = generate_citations(query_text)

    # Use " " (single space) when no citations found, otherwise join with ";"
    predicted = ";".join(raw_citations) if raw_citations else " "

    predictions.append({
        "query_id": query_id,
        "predicted_citations": predicted,
    })

print(f"\nGenerated predictions for {len(predictions)} queries")

Generating predictions:   0%|          | 0/10 [00:00<?, ?it/s]

Raw output: ["Art. 221 Abs. 1 lit. b StPO", "Art. 212 Abs. 2 StPO", "Art. 5 Ziff. 3 EMRK", "Art. 36 BV", "BGE 137 IV 122 E. 3.2", "BGE 143 IV 160 E. 2.2", "BGE 140 IV 19 E. 2.1", "BGE 139 IV 41 E. 2.2.1"]

Raw output: ["Art. 17 LAI", "Art. 8 LAI", "Art. 7 LAI", "Art. 28 LAI", "Art. 6 ATSG", "Art. 7 ATSG", "BGE 143 V 409 E. 4.5", "BGE 141 V 281 E. 2.2.1", "BGE 140 V 193 E. 3.2", "BGE 137 V 210 E. 1.2.1", "BGE 130 V 352 E. 3.2", "BGE 129 V 167 E. 1", "BGE 127 V 294 E. 4c", "BGE 126 V 75 E. 2b", "BGE 115 V 133 E. 2"]

Raw output: ["Art. 221 Abs. 1 StPO", "Art. 31 Abs. 3 BV", "Art. 10 Abs. 2 StPO", "Art. 29 Abs. 2 BV", "BGE 137 IV 122 E. 3.2", "BGE 143 IV 160 E. 2.1", "BGE 137 IV 122 E. 3.1", "BGE 140 IV 192 E. 2.1", "BGE 133 I 270 E. 3.2", "BGE 124 I 208 E. 3"]

Raw output: ["Art. 505 ZGB", "Art. 470 ZGB", "Art. 519 ZGB", "Art. 520 ZGB", "Art. 467 ZGB", "BGE 132 III 305 E. 3.2", "BGE 138 III 150 E. 2.1", "BGE 117 II 231 E. 2b", "BGE 102 II 289 E. 2"]

Raw output: ["Art. 133 ZGB", "Art. 29

In [8]:
# Preview predictions
predictions_df = pd.DataFrame(predictions)
predictions_df.head(10)

,query_id,predicted_citations
0,val_001,Art. 221 Abs. 1 lit. b STPO;Art. 212 Abs. 2 ST...
1,val_002,Art. 17 LAI;Art. 8 LAI;Art. 7 LAI;Art. 28 LAI;...
2,val_003,Art. 221 Abs. 1 STPO;Art. 31 Abs. 3 BV;Art. 10...
3,val_004,Art. 505 ZGB;Art. 470 ZGB;Art. 519 ZGB;Art. 52...
4,val_005,Art. 133 ZGB;Art. 296 ZPO;Art. 314 ZGB;Art. 31...
5,val_006,Art. 41 OR;Art. 97 OR;Art. 364 Abs. 1 OR;Art. ...
6,val_007,Art. 239 OR;Art. 242 OR;Art. 243 OR;Art. 957 Z...
7,val_008,Art. 158 STGB;Art. 12 STGB;Art. 29 BV;Art. 9 B...
8,val_009,Art. 276 ZGB;Art. 277 ZGB;Art. 285 ZGB;Art. 28...
9,val_010,Art. 398 OR;Art. 97 OR;Art. 100 OR;Art. 127 OR...


## 6. Create Submission

In [9]:
# Save submission
submission_path = OUTPUT_PATH / "submission.csv"
predictions_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total predictions: {len(predictions_df)}")

# Show sample
print("\nSample submission:")
print(predictions_df.head())

Submission saved to: /workspaces/lexcite/output/submission.csv
Total predictions: 10

Sample submission:
  query_id                                predicted_citations
0  val_001  Art. 221 Abs. 1 lit. b STPO;Art. 212 Abs. 2 ST...
1  val_002  Art. 17 LAI;Art. 8 LAI;Art. 7 LAI;Art. 28 LAI;...
2  val_003  Art. 221 Abs. 1 STPO;Art. 31 Abs. 3 BV;Art. 10...
3  val_004  Art. 505 ZGB;Art. 470 ZGB;Art. 519 ZGB;Art. 52...
4  val_005  Art. 133 ZGB;Art. 296 ZPO;Art. 314 ZGB;Art. 31...


## Summary

This baseline notebook demonstrates a simple direct generation approach:

1. **Prompt engineering**: We use a structured prompt that asks the LLM to generate Swiss legal citations in standard format.

2. **Citation normalization**: The generated citations are normalized to canonical form for consistent evaluation.

3. **Limitations**:
   - The LLM may hallucinate non-existent citations
   - No access to actual legal documents for verification
   - Relies entirely on the LLM's training data knowledge

For better results, see the **Agentic Retrieval Baseline** notebook which uses search tools to ground the generation in actual legal documents.